# End-to-End Fine-Tuning with BNP Paribas Cardif Claims

Fine-tunes a KumoTabular model on the BNP Paribas Cardif Claims dataset, then benchmarks pretrained vs finetuned inference on the same test set.

In [0]:
%pip install openml git+https://github.com/NVIDIA/structured-data-models.git --quiet -q

In [0]:
dbutils.library.restartPython()

## Config

In [0]:
import importlib.util
import os
import sys

# Make the repo-level common/ package importable.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
COMMON_PATH = os.path.join(REPO_ROOT, "common")
CONFIG_PATH = os.path.join(COMMON_PATH, "config.py")

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"Could not find config.py at {CONFIG_PATH}")

spec = importlib.util.spec_from_file_location("project_config", CONFIG_PATH)
project_config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(project_config)
CATALOG = project_config.CATALOG

SCHEMA = "default"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

current_user = spark.sql("SELECT current_user()").collect()[0][0]
MLFLOW_EXPERIMENT_NAME = f"/Users/{current_user}/tabular-fm-databricks-finetune"

print(f"Catalog/schema: {CATALOG}.{SCHEMA}")
print(f"MLflow experiment: {MLFLOW_EXPERIMENT_NAME}")

In [0]:
import gc, time, os

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss

import mlflow
import torch
import sdm

import openml



In [0]:
# Download BNP Paribas Cardif Claims Management dataset from OpenML
dataset = openml.datasets.get_dataset(46625)
df, y, _, _ = dataset.get_data(dataset_format="dataframe")

# Cast unsigned int types to signed (Spark doesn't support uint via Arrow)
for col in df.select_dtypes(include=[np.uint8, np.uint16, np.uint32, np.uint64]).columns:
    df[col] = df[col].astype(np.int32)
# Cast remaining non-float numeric to float64 for safety
for col in df.select_dtypes(include=["category"]).columns:
    df[col] = df[col].astype(str)

print(f"Dataset: {dataset.name}")
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}, Columns: {df.shape[1]}")

# Keep only rows with a target value, then split (80/20)
df_labeled = df.dropna(subset=["target"])
print(f"Labeled rows: {len(df_labeled):,} (dropped {len(df) - len(df_labeled):,} with null target)")

df_train, df_score = train_test_split(df_labeled, test_size=0.2, random_state=42)

print(f"\nTrain: {df_train.shape[0]:,} rows")
print(f"Score: {df_score.shape[0]:,} rows")

# Save to Unity Catalog
TABLE_TRAIN = "bnp_paribas_cardif_train"
TABLE_SCORE = "bnp_paribas_cardif_score"

spark.createDataFrame(df_train).write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE_TRAIN}")
print(f"\n✓ Saved {df_train.shape[0]:,} rows to {CATALOG}.{SCHEMA}.{TABLE_TRAIN}")

spark.createDataFrame(df_score).write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE_SCORE}")
print(f"✓ Saved {df_score.shape[0]:,} rows to {CATALOG}.{SCHEMA}.{TABLE_SCORE}")

In [0]:
N_TRAIN_SAMPLE = None
N_TEST_SAMPLE = None
N_FEATURES = None

TRAIN_TABLE = f"{CATALOG}.{SCHEMA}.bnp_paribas_cardif_train"
SCORE_TABLE = f"{CATALOG}.{SCHEMA}.bnp_paribas_cardif_score"
TARGET_COL = "target"
ID_COL = "ID"

# Load data from UC
train_pdf = spark.table(TRAIN_TABLE).toPandas()
score_pdf = spark.table(SCORE_TABLE).toPandas()

# Filter out rows with null target
train_fit_pdf = train_pdf.dropna(subset=[TARGET_COL])
score_eval_pdf = score_pdf.dropna(subset=[TARGET_COL])

# Feature columns (exclude ID and target)
all_feature_cols = [c for c in train_fit_pdf.columns if c not in (ID_COL, TARGET_COL)]
feature_cols = all_feature_cols if N_FEATURES is None else all_feature_cols[:N_FEATURES]

# Sample rows
train_sample = train_fit_pdf if N_TRAIN_SAMPLE is None else train_fit_pdf.sample(n=min(N_TRAIN_SAMPLE, len(train_fit_pdf)), random_state=42)
test_sample  = score_eval_pdf if N_TEST_SAMPLE is None else score_eval_pdf.sample(n=min(N_TEST_SAMPLE, len(score_eval_pdf)), random_state=42)

X_train = train_sample[feature_cols].reset_index(drop=True)
y_train = train_sample[[TARGET_COL]].reset_index(drop=True)
X_test  = test_sample[feature_cols].reset_index(drop=True)
y_test  = test_sample[TARGET_COL].values.astype(float)

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Train: {len(X_train):,} rows (sampled from {len(train_fit_pdf):,})")
print(f"Test:  {len(X_test):,} rows (sampled from {len(score_eval_pdf):,})")
print(f"Features: {len(feature_cols)} of {len(all_feature_cols)} total")
print(f"Device: {device}")

In [0]:
VOLUME_NAME = "finetuned_models"
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME_NAME}")
volume_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"
print(f"✓ Volume ready: {volume_path}")

## Fine-tuning

In [0]:
import gc, torch, subprocess

# 1) Delete known GPU-heavy variables by name (covers dicts of tensors,
#    closures capturing models, sdm objects, optimizer state, etc.)
_gpu_vars = [
    'model', 'model_ft', 'classifier', 'ft_classifier',
    'pretrained_model', 'finetuned_model',
    'train_table', 'val_table', 'table',
    'X_train_tensor', 'y_train_tensor', 'X_test_tensor',
    'x_ctx', 'y_ctx', 'x_test', 'y_true_tensor',
    'optimizer', 'scaler', 'best_state',
    'evaluate', 'train_recipe', 'extract_predictions',
    'pred_pre', 'pred_ft', 'pred_pdf', 'pred_pdf_ft',
    'chronos_pipeline',
]
for _v in _gpu_vars:
    if _v in globals():
        del globals()[_v]

# 2) Sweep remaining torch objects (catches anything we missed above)
for _name in list(globals()):
    _obj = globals()[_name]
    if isinstance(_obj, (torch.Tensor, torch.nn.Module, torch.optim.Optimizer, torch.amp.GradScaler)):
        del globals()[_name]
        continue
    try:
        if hasattr(_obj, 'device') and 'cuda' in str(_obj.device):
            del globals()[_name]
    except Exception:
        pass

gc.collect()
torch.cuda.empty_cache()

print(f"PyTorch allocated: {torch.cuda.memory_allocated()/1e6:.0f} MiB")
print(f"PyTorch reserved:  {torch.cuda.memory_reserved()/1e6:.0f} MiB")
print(subprocess.check_output(['nvidia-smi'], text=True))

In [0]:
import torch.nn.functional as F
import sdm.processing as sp

# ── Fine-tuning configuration ───────────────────────────────────────
FT_MODEL_SIZE = "small"
FT_LR = 1e-2#1e-5                   # Above ~5e-4 destabilises pretrained weights
FT_MAX_EPOCHS = 50              # Start with 1; increase to 50+ for best results
FT_PATIENCE = 10                # Use 10 for real runs
FT_STEPS_PER_EPOCH = 30
FT_CONTEXT_SIZE = 1024 #256
FT_QUERY_SIZE = 256 #128
FT_NUM_ESTIMATORS_TRAIN = 2    # Effective batch dimension
FT_NUM_ESTIMATORS_VAL = 2
FT_NUM_ESTIMATORS_INFERENCE = 8
FT_EVAL_CONTEXT_SIZE = 1024    # Subsample context rows for evaluation
FT_SEED = 0

torch.manual_seed(FT_SEED)

# ── Prepare data (uses train_fit_pdf & feature_cols from cell 9) ────
ft_cols = feature_cols + [TARGET_COL]
ft_df = train_fit_pdf[ft_cols].reset_index(drop=True)

stypes = sdm.infer_stypes(ft_df, overrides={TARGET_COL: "categorical"})
table = sdm.TableTensor.from_pandas(ft_df, stypes=stypes, device=device)

train_idx, val_idx = train_test_split(
    torch.arange(len(ft_df)).numpy(),
    test_size=0.2,
    random_state=FT_SEED,
    stratify=ft_df[TARGET_COL],
)
train_table = table[torch.from_numpy(train_idx).to(device)]
val_table = table[torch.from_numpy(val_idx).to(device)]
del table
gc.collect()
torch.cuda.empty_cache()

print(f"Train: {len(train_idx):,} rows | Val: {len(val_idx):,} rows")
print(f"Features: {len(feature_cols)} | Device: {device}")

# ── Model & recipe ──────────────────────────────────────────────────
model = sdm.models.KumoTabular(
    task="classification", size=FT_MODEL_SIZE, device=device, pretrained=True,
)

train_recipe = model.default_recipe()
train_recipe.target = sp.StypeDispatch(
    categorical=[
        sp.AlignCategories(),
        sp.ShuffleCategories(method="shift"),
    ],
    numerical=sp.Standardize(),
)
train_recipe.output = sp.Identity()


def evaluate(context, query):
    """Evaluate on full query set using a fixed subsample of context."""
    model.eval()
    with torch.no_grad():
        idx = torch.randperm(context.size(0), device=device)[:FT_EVAL_CONTEXT_SIZE]
        ctx = context[idx]
        with torch.amp.autocast('cuda', dtype=torch.float16):
            out = model(
                x_context=ctx.drop_columns(TARGET_COL),
                y_context=ctx[TARGET_COL],
                x_query=query.drop_columns(TARGET_COL),
                num_estimators=FT_NUM_ESTIMATORS_VAL,
            )
    prob, y = sdm.evaluation.to_class_indices(
        out, query[TARGET_COL], missing_score=0.0,
    )
    prob = torch.softmax(prob, dim=-1)
    y_pred = prob.argmax(dim=-1).cpu().numpy()
    y_true = y.cpu().numpy()
    y_prob = prob[:, -1].cpu().numpy()
    del out, prob, ctx
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, average="binary")),
        "log_loss": float(log_loss(y_true, y_prob, labels=[0, 1])),
    }


# ── Training loop with MLflow ───────────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=FT_LR)
scaler = torch.amp.GradScaler('cuda')
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name="bnp_paribas_finetuning_process"):
    run_id = mlflow.active_run().info.run_id
    ckpt_path = f"{volume_path}/kumo_tabular_{FT_MODEL_SIZE}_classification_finetuned_{run_id}.pt"

    mlflow.log_params({
        "vendor": "kumo",
        "model_type": "KumoTabular",
        "model_size": FT_MODEL_SIZE,
        "task": "classification",
        "dataset": "bnp_paribas_cardif",
        "lr": FT_LR,
        "max_epochs": FT_MAX_EPOCHS,
        "patience": FT_PATIENCE,
        "steps_per_epoch": FT_STEPS_PER_EPOCH,
        "context_size": FT_CONTEXT_SIZE,
        "query_size": FT_QUERY_SIZE,
        "num_estimators_train": FT_NUM_ESTIMATORS_TRAIN,
        "num_estimators_val": FT_NUM_ESTIMATORS_VAL,
        "num_estimators_inference": FT_NUM_ESTIMATORS_INFERENCE,
        "eval_context_size": FT_EVAL_CONTEXT_SIZE,
        "n_features": len(feature_cols),
        "n_train": len(train_idx),
        "n_val": len(val_idx),
        "seed": FT_SEED,
        "device": device,
    })

    # Baseline before any training
    val_metrics = evaluate(context=train_table, query=val_table)
    print(f"epoch=0/{FT_MAX_EPOCHS} val_acc={val_metrics['accuracy']:.4f} val_f1={val_metrics['f1']:.4f} val_logloss={val_metrics['log_loss']:.4f}")
    mlflow.log_metrics({f"ft_per_epoch_val_{k}": v for k, v in val_metrics.items()}, step=0)

    best_val_metrics = val_metrics.copy()
    epochs_without_improvement = 0
    torch.save(model.state_dict(), ckpt_path)

    # Free evaluation residuals before training starts
    gc.collect()
    torch.cuda.empty_cache()

    t_start = time.time()
    for epoch in range(1, FT_MAX_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for _ in range(FT_STEPS_PER_EPOCH):
            index = torch.randperm(train_table.size(0), device=device)
            index = index[:FT_CONTEXT_SIZE + FT_QUERY_SIZE]
            context = train_table[index[:FT_CONTEXT_SIZE]]
            query = train_table[index[FT_CONTEXT_SIZE:]]

            optimizer.zero_grad()
            with torch.amp.autocast('cuda', dtype=torch.float16):
                out = model(
                    x_context=context.drop_columns(TARGET_COL),
                    y_context=context[TARGET_COL],
                    x_query=query.drop_columns(TARGET_COL),
                    recipe=train_recipe,
                    num_estimators=FT_NUM_ESTIMATORS_TRAIN,
                )[0]
                logits, y = sdm.evaluation.to_class_indices(
                    out, query[TARGET_COL], missing_score=-torch.inf,
                )
                loss = F.cross_entropy(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += float(loss.detach())
            del context, query, out, logits, y, loss, index
            torch.cuda.empty_cache()

        avg_loss = total_loss / FT_STEPS_PER_EPOCH
        # Free training activations before evaluation
        gc.collect()
        torch.cuda.empty_cache()

        val_metrics = evaluate(context=train_table, query=val_table)

        # Free evaluation residuals before next training epoch
        gc.collect()
        torch.cuda.empty_cache()

        print(
            f"epoch={epoch}/{FT_MAX_EPOCHS} "
            f"val_acc={val_metrics['accuracy']:.4f} "
            f"val_f1={val_metrics['f1']:.4f} "
            f"val_logloss={val_metrics['log_loss']:.4f} "
            f"train_loss={avg_loss:.4f}"
        )
        mlflow.log_metrics({f"ft_per_epoch_val_{k}": v for k, v in val_metrics.items()}, step=epoch)
        mlflow.log_metric("ft_per_epoch_train_loss", avg_loss, step=epoch)

        if val_metrics["accuracy"] > best_val_metrics["accuracy"]:
            best_val_metrics = val_metrics.copy()
            epochs_without_improvement = 0
            torch.save(model.state_dict(), ckpt_path)
            print(f"  → New best! Saved checkpoint.")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= FT_PATIENCE:
                print(f"  → Early stopping after {FT_PATIENCE} epochs without improvement.")
                break

    ft_time = time.time() - t_start

    # Load best checkpoint
    best_state = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(best_state)

    mlflow.log_metrics({
        "ft_best_val_acc": best_val_metrics["accuracy"],
        "ft_best_val_f1": best_val_metrics["f1"],
        "ft_best_val_log_loss": best_val_metrics["log_loss"],
        "ft_finetune_time_s": ft_time,
        "ft_epochs_completed": min(epoch, FT_MAX_EPOCHS),
    })
    mlflow.log_artifact(ckpt_path, artifact_path="checkpoints")

print(f"\n✓ Fine-tuning complete (run {run_id})")
print(f"  Best val accuracy: {best_val_metrics['accuracy']:.4f}")
print(f"  Best val f1:       {best_val_metrics['f1']:.4f}")
print(f"  Best val log_loss: {best_val_metrics['log_loss']:.4f}")
print(f"  Time: {ft_time:.1f}s")
print(f"  Checkpoint: {ckpt_path}")

## Inference — Pretrained Model

In [0]:
# ── Clean GPU ───────────────────────────────────────────────────────
import gc, torch, subprocess

_gpu_vars = [
    'model', 'model_ft', 'classifier', 'ft_classifier',
    'pretrained_model', 'finetuned_model',
    'train_table', 'val_table', 'table',
    'X_train_tensor', 'y_train_tensor', 'X_test_tensor',
    'x_ctx', 'y_ctx', 'x_test', 'y_true_tensor',
    'optimizer', 'scaler', 'best_state',
    'evaluate', 'train_recipe', 'extract_predictions',
    'pred_pre', 'pred_ft', 'pred_pdf', 'pred_pdf_ft',
]
for _v in _gpu_vars:
    if _v in globals():
        del globals()[_v]
for _name in list(globals()):
    _obj = globals()[_name]
    if isinstance(_obj, (torch.Tensor, torch.nn.Module, torch.optim.Optimizer, torch.amp.GradScaler)):
        del globals()[_name]
        continue
    try:
        if hasattr(_obj, 'device') and 'cuda' in str(_obj.device):
            del globals()[_name]
    except Exception:
        pass
gc.collect()
torch.cuda.empty_cache()
print(f"GPU cleaned: {torch.cuda.memory_allocated()/1e6:.0f} MiB allocated")

# ── Score pretrained model ──────────────────────────────────────────
classifier = sdm.models.KumoTabular(task="classification", device=device, size="small", pretrained=True)

X_train_tensor = sdm.TableTensor.from_pandas(df=X_train, stypes=sdm.infer_stypes(X_train), device=device)
y_train_tensor = sdm.TableTensor.from_pandas(df=y_train, stypes=sdm.infer_stypes(y_train, overrides={TARGET_COL: "categorical"}), device=device)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name="bnp_paribas_cardif_kumo_classification"):
    mlflow.log_params({
        "vendor": "kumo",
        "model_type": "KumoTabularClassifier",
        "task": "bnp_paribas_cardif",
        "problem_type": "Classification",
        "finetuned": False,
        "train_table": TRAIN_TABLE,
        "score_table": SCORE_TABLE,
        "target_col": TARGET_COL,
        "n_features": len(feature_cols),
        "n_features_total": len(all_feature_cols),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "device": device,
        "size": "small",
        "num_estimators": 1,
    })

    t0 = time.time()
    classifier.fit(x=X_train_tensor, y=y_train_tensor, num_estimators=1)
    fit_time = time.time() - t0

    del X_train_tensor, y_train_tensor
    torch.cuda.empty_cache()

    test_stypes = sdm.infer_stypes(X_test)
    t1 = time.time()
    X_test_tensor = sdm.TableTensor.from_pandas(df=X_test, stypes=test_stypes, device=device)
    pred_pdf = classifier.predict(X_test_tensor).to_pandas()
    predict_time = time.time() - t1

    y_pred = pred_pdf.idxmax(axis=1).astype(float).astype(int).values

    results_pdf = test_sample[[ID_COL, TARGET_COL]].copy().reset_index(drop=True)
    results_pdf["predicted"] = y_pred
    for col in pred_pdf.columns:
        results_pdf[f"prob_{col}"] = pred_pdf[col].astype(float).values

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="binary")
    col_1 = [c for c in pred_pdf.columns if float(c) == 1.0][0]
    y_prob = pred_pdf[col_1].astype(float).values
    auc = roc_auc_score(y_test, y_prob)
    logloss = log_loss(y_test, y_prob)

    mlflow.log_metrics({"accuracy": accuracy, "f1": f1, "auc": auc, "log_loss": logloss, "fit_time_s": fit_time, "predict_time_s": predict_time})
    mlflow.log_table(results_pdf, "score_predictions.json")

    pretrained_run_id = mlflow.active_run().info.run_id

print(f"Run ID: {pretrained_run_id}")
print(f"Fit time:     {fit_time:.1f}s")
print(f"Predict time: {predict_time:.1f}s")
print(f"Accuracy: {accuracy:.6f}")
print(f"F1:       {f1:.6f}")
print(f"AUC:      {auc:.6f}")
print(f"Log Loss: {logloss:.6f}")
display(results_pdf.head(10))

## Inference — Finetuned Model

In [0]:
# ── Clean GPU ───────────────────────────────────────────────────────
import gc, torch, subprocess

_gpu_vars = [
    'model', 'model_ft', 'classifier', 'ft_classifier',
    'pretrained_model', 'finetuned_model',
    'train_table', 'val_table', 'table',
    'X_train_tensor', 'y_train_tensor', 'X_test_tensor',
    'x_ctx', 'y_ctx', 'x_test', 'y_true_tensor',
    'optimizer', 'scaler', 'best_state',
    'evaluate', 'train_recipe', 'extract_predictions',
    'pred_pre', 'pred_ft', 'pred_pdf', 'pred_pdf_ft',
]
for _v in _gpu_vars:
    if _v in globals():
        del globals()[_v]
for _name in list(globals()):
    _obj = globals()[_name]
    if isinstance(_obj, (torch.Tensor, torch.nn.Module, torch.optim.Optimizer, torch.amp.GradScaler)):
        del globals()[_name]
        continue
    try:
        if hasattr(_obj, 'device') and 'cuda' in str(_obj.device):
            del globals()[_name]
    except Exception:
        pass
gc.collect()
torch.cuda.empty_cache()
print(f"GPU cleaned: {torch.cuda.memory_allocated()/1e6:.0f} MiB allocated")

# ── Load finetuned model
ft_classifier = sdm.models.KumoTabular(
    task="classification", device=device, size=FT_MODEL_SIZE, pretrained=False,
)
ft_classifier.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
ft_classifier.eval()

# Use the exact same train/test data as cell 12
X_train_tensor = sdm.TableTensor.from_pandas(df=X_train, stypes=sdm.infer_stypes(X_train), device=device)
y_train_tensor = sdm.TableTensor.from_pandas(df=y_train, stypes=sdm.infer_stypes(y_train, overrides={TARGET_COL: "categorical"}), device=device)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name="bnp_paribas_cardif_kumo_classification_finetuned"):
    mlflow.log_params({
        "vendor": "kumo",
        "model_type": "KumoTabularClassifier",
        "task": "bnp_paribas_cardif",
        "problem_type": "Classification",
        "finetuned": True,
        "finetune_checkpoint": ckpt_path,
        "finetune_run_id": run_id,
        "train_table": TRAIN_TABLE,
        "score_table": SCORE_TABLE,
        "target_col": TARGET_COL,
        "n_features": len(feature_cols),
        "n_features_total": len(all_feature_cols),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "device": device,
        "size": FT_MODEL_SIZE,
        "num_estimators": 1,
    })

    # Fit (stores context for ICL prediction — same data as zero-shot)
    t0 = time.time()
    ft_classifier.fit(x=X_train_tensor, y=y_train_tensor, num_estimators=1)
    fit_time = time.time() - t0

    del X_train_tensor, y_train_tensor
    torch.cuda.empty_cache()

    # Predict on the same test set
    test_stypes = sdm.infer_stypes(X_test)
    t1 = time.time()
    X_test_tensor = sdm.TableTensor.from_pandas(df=X_test, stypes=test_stypes, device=device)
    pred_pdf_ft = ft_classifier.predict(X_test_tensor).to_pandas()
    predict_time = time.time() - t1

    y_pred_ft = pred_pdf_ft.idxmax(axis=1).astype(float).astype(int).values

    results_pdf_ft = test_sample[[ID_COL, TARGET_COL]].copy().reset_index(drop=True)
    results_pdf_ft["predicted"] = y_pred_ft
    for col in pred_pdf_ft.columns:
        results_pdf_ft[f"prob_{col}"] = pred_pdf_ft[col].astype(float).values

    # Score
    accuracy_ft = accuracy_score(y_test, y_pred_ft)
    f1_ft = f1_score(y_test, y_pred_ft, average="binary")
    col_1 = [c for c in pred_pdf_ft.columns if float(c) == 1.0][0]
    y_prob_ft = pred_pdf_ft[col_1].astype(float).values
    auc_ft = roc_auc_score(y_test, y_prob_ft)
    logloss_ft = log_loss(y_test, y_prob_ft)

    mlflow.log_metrics({"accuracy": accuracy_ft, "f1": f1_ft, "auc": auc_ft, "log_loss": logloss_ft, "fit_time_s": fit_time, "predict_time_s": predict_time})
    mlflow.log_table(results_pdf_ft, "score_predictions.json")

    ft_run_id = mlflow.active_run().info.run_id

print(f"Run ID: {ft_run_id}")
print(f"Fit time:     {fit_time:.1f}s")
print(f"Predict time: {predict_time:.1f}s")
print(f"Accuracy: {accuracy_ft:.6f}")
print(f"F1:       {f1_ft:.6f}")
print(f"AUC:      {auc_ft:.6f}")
print(f"Log Loss: {logloss_ft:.6f}")
display(results_pdf_ft.head(10))